In [0]:
#Using the UDF in SQL queries
# You already know using UDFs with df using '.withColumn() 

In [0]:
from pyspark.sql import Row

# Employee performance data
employee_data = [
    Row(emp_id=101, emp_name="Alice Johnson", performance_score=92, years_service=5),
    Row(emp_id=102, emp_name="Bob Smith", performance_score=78, years_service=3),
    Row(emp_id=103, emp_name="Carol White", performance_score=55, years_service=2),
    Row(emp_id=104, emp_name="David Brown", performance_score=85, years_service=7),
    Row(emp_id=105, emp_name="Eve Davis", performance_score=95, years_service=4),
    Row(emp_id=106, emp_name="Frank Miller", performance_score=68, years_service=6),
]

employees_df = spark.createDataFrame(employee_data)
display(employees_df)

In [0]:
from pyspark.sql.functions import udf

@udf(returnType = "double")
def calculate_bonus(score, years):
    if score is None or years is None:
        return None
    
    base_bonus = score *10
    if years>=5:
        loyalty_bonus = base_bonus*1.2
        return loyalty_bonus
    else:
        return base_bonus



In [0]:
from pyspark.sql.functions import col
employees_with_rating = employees_df.withColumn("performance_bonus",
    calculate_bonus(
        col("performance_score"),
        col("years_service")) )

In [0]:
employees_df_bonus_calc.display()

In [0]:
'''
#ok exercise 3 : UDFs in SQL query

#Step 1: Register the UDF
spark.udf.register("function_name_in_sql", udf_function)

#Step 2 : create temporay sql view of the dataframe
df.createOrReplaceTempView("table_name")

#Step 3 : Use the function in SQL
SELECT * , function_name_in_sql(col1,col2)
FROM table_name

'''

In [0]:
#step 1 :
#register the UDF for SQL
spark.udf.register("calculate_bonus_sql", calculate_bonus)


#step 2 :
#create a temporary view
employees_with_rating.createOrReplaceTempView("employees_table")

In [0]:
sql_result = spark.sql(
""" 
SELECT *, 
calculate_bonus_sql(performance_score, years_service) AS bonus_amount
FROM employees_table
ORDER BY emp_id 
"""
)

In [0]:
sql_result.display()

#### Customer Risk Score

- logic to calcualte risk
- Amount spend, Tenure trsust factor -->final score

In [0]:
# Transaction data for fraud risk analysis
from pyspark.sql import Row
risk_data = [
    Row(transaction_id=2001, customer_id=601, purchase_amount=35.00, customer_tenure_years=0.3),
    Row(transaction_id=2002, customer_id=602, purchase_amount=120.00, customer_tenure_years=1.0),
    Row(transaction_id=2003, customer_id=603, purchase_amount=750.00, customer_tenure_years=0.2),
    Row(transaction_id=2004, customer_id=604, purchase_amount=280.00, customer_tenure_years=3.5),
    Row(transaction_id=2005, customer_id=605, purchase_amount=55.00, customer_tenure_years=0.8),
    Row(transaction_id=2006, customer_id=606, purchase_amount=900.00, customer_tenure_years=5.0),
    Row(transaction_id=2007, customer_id=607, purchase_amount=180.00, customer_tenure_years=0.4),
    Row(transaction_id=2008, customer_id=608, purchase_amount=420.00, customer_tenure_years=2.5),
]

risk_df = spark.createDataFrame(risk_data)
display(risk_df)

In [0]:
from pyspark.sql.functions import pandas_udf,col
import pandas as pd

@pandas_udf("double")  #pandas decorator
def calculate_risk_score(amounts: pd.Series, tenure:pd.Series) -> pd.Series:
    
    amount_risk_mask1 = amounts<=50
    amount_risk_mask2 = (amounts >50) & (amounts <=200)
    amount_risk_mask3 = (amounts>200) & (amounts <=500)
    amount_risk_mask4 = amounts >500

    base_amount_risk = (amount_risk_mask1 *0) + (amount_risk_mask2 * 10) + (amount_risk_mask3 * 20) + (amount_risk_mask4 * 40)

    Tenure_trust_factor1 = tenure<=1
    Tenure_trust_factor2 = (tenure >1) & (tenure <=2)
    Tenure_trust_factor3 = tenure > 2

    base_tenure_factor = (Tenure_trust_factor1*1)\
    +(Tenure_trust_factor2*.8)\
    +(Tenure_trust_factor3*.5)

    Final_risk_score = base_amount_risk *base_tenure_factor
    return Final_risk_score



risk_with_scores = risk_df.withColumn("risk_score",calculate_risk_score(col("purchase_amount"),col("customer_tenure_years")))

    





In [0]:
risk_with_scores.display()